# 03.1 序列基础（Sequence Basics）

sequence data 的最基本表示搞清楚。  

进入 NLP 或时间序列之前，你至少要先理解：  

- 词元或离散符号（token）
- 词元编号（token id）
- 词表（vocabulary）
- 序列长度（sequence length）
- 嵌入向量（embedding）
- 填充（padding）
- 掩码（mask）

如果这些概念是模糊的，后面的 `LSTM` 和 `Attention` 很容易看晕。  


## 学习目标

学完后你应该能

1. 理解离散 token 如何变成整数 id
2. 理解序列 batch 的常见 shape
3. 使用 `nn.Embedding` 把 token id 映射成 embedding
4. 理解为什么需要 padding
5. 构造 padding mask
6. 为后续 `LSTM` 和 `Attention` 做 shape 准备

In [ ]:
import torch
import torch.nn as nn

## 1. Token、词表与 id 映射

最开始的序列常常是字符串或离散符号。  

模型不能直接吃字符串，所以通常要先映射到整数 id。  


In [ ]:
sentences = [
    ["i", "like", "pytorch"],
    ["you", "like", "deep", "learning"],
    ["i", "study"],
]

special_tokens = ["<pad>", "<unk>"]
vocab = special_tokens + sorted({token for sent in sentences for token in sent})
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

print("vocab =", vocab)
print("stoi =", stoi)
print("itos[2] =", itos[2])

In [ ]:
encoded_sentences = [[stoi.get(token, stoi["<unk>"]) for token in sent] for sent in sentences]

print("original sentences / 原始句子:", sentences)
print("encoded sentences / 编码后句子:", encoded_sentences)

## 2. 序列长度

序列模型里最常见的一个维度就是 `sequence length`。  

例如

- `['i', 'like', 'pytorch']` 的长度是 3
- `['you', 'like', 'deep', 'learning']` 的长度是 4

问题是：同一个 batch 里的序列长度经常不一样。  


## 3. 为什么需要 padding

因为张量要求一个 batch 内部形状统一，所以需要把短序列补到相同长度。  

通常用 `<pad>` token 填充。  


In [ ]:
pad_id = stoi["<pad>"]
max_len = max(len(seq) for seq in encoded_sentences)
padded_sentences = [seq + [pad_id] * (max_len - len(seq)) for seq in encoded_sentences]

batch_ids = torch.tensor(padded_sentences, dtype=torch.long)

print("max_len =", max_len)
print("padded_sentences =", padded_sentences)
print("batch_ids =\n", batch_ids)
print("batch_ids.shape =", batch_ids.shape)

这里 `batch_ids.shape == (3, 4)` 表示：  

- `3` 个样本
- 每个样本补到了长度 `4`

在很多 PyTorch 序列模型里，我们常用 `batch_first=True`，所以 shape 写成：  

- `(batch_size, seq_len)`
- `(batch_size, seq_len, embedding_dim)`

In [ ]:
# 练习 1
# 给定下面三条序列 id，请把它们 padding 到相同长度。
# Given the three sequences below, pad them to the same length.

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0

# max_len =
# padded =
# batch =
# print(batch)
# print(batch.shape)

In [ ]:
# 练习 1 参考答案

seqs = [[3, 5], [2, 4, 6, 7], [9]]
pad_id = 0
max_len = max(len(seq) for seq in seqs)
padded = [seq + [pad_id] * (max_len - len(seq)) for seq in seqs]
batch = torch.tensor(padded, dtype=torch.long)
print(batch)
print(batch.shape)

## 4. `nn.Embedding` / `nn.Embedding`

token id 本身只是编号，不包含语义。  

`Embedding` 的作用是把每个 token id 映射成一个稠密向量。  


In [ ]:
vocab_size = len(vocab)
embedding_dim = 6
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

embedded = embedding(batch_ids)

print("batch_ids.shape =", batch_ids.shape)
print("embedded.shape =", embedded.shape)

这里的 shape 从 `(batch_size, seq_len)` 变成了 `(batch_size, seq_len, embedding_dim)`。  

也就是说，每个 token 都变成了一个向量。  


In [ ]:
print("batch_ids[0] =", batch_ids[0])
print("embedded[0].shape =", embedded[0].shape)
print("embedded[0] =\n", embedded[0])

In [ ]:
# 练习 2
# 构造一个 embedding 层，词表大小 20，embedding 维度 8。
# Build an embedding layer with vocab size 20 and embedding dimension 8.
#
# 然后把一个 shape 为 (4, 5) 的 token id batch 喂进去，打印输出 shape。
# Then feed it a token-id batch of shape (4, 5) and print the output shape.

# emb =
# batch =
# out =
# print(out.shape)

In [ ]:
# 练习 2 参考答案

emb = nn.Embedding(20, 8)
batch = torch.randint(low=0, high=20, size=(4, 5))
out = emb(batch)
print(out.shape)

## 5. Padding Mask / Padding Mask

虽然 padding 让 batch 形状统一了，但 `<pad>` 并不是真实内容。  

所以很多序列模型需要知道：哪些位置是 padding。  

这就是 `padding mask` 的作用。  


In [ ]:
padding_mask = batch_ids == pad_id

print("batch_ids =\n", batch_ids)
print("padding_mask =\n", padding_mask)
print("padding_mask.shape =", padding_mask.shape)

这里 `True` 表示当前位置是 padding。  

这类 mask 在后面的 `Attention` notebook 里会非常重要。  


In [ ]:
# 练习 3
# 给定下面的 batch ids，构造 padding mask。
# Given the batch ids below, construct the padding mask.

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0

# mask =
# print(mask)

In [ ]:
# 练习 3 参考答案

batch = torch.tensor([
    [4, 5, 0, 0],
    [1, 2, 3, 0],
    [7, 8, 9, 1],
])
pad_id = 0
mask = batch == pad_id
print(mask)

## 6. 常见 shape 对照表

你应该尽快习惯下面这组 shape：  

- token ids: `(batch_size, seq_len)`
- embeddings: `(batch_size, seq_len, embedding_dim)`
- padding mask: `(batch_size, seq_len)`

后面只是在这些基础上增加新的维度语义。  


## 7. 小结

这一节的核心不是 API 数量，而是把序列输入格式彻底想清楚。  

你现在应该能回答

1. 为什么 token 常要先映射成整数 id？
2. 为什么同一个 batch 里需要 padding？
3. `Embedding` 为什么会把 shape 从 `(B, T)` 变成 `(B, T, D)`？
4. `padding mask` 的作用是什么？

下一步建议

- 进入 `LSTM` notebook，观察这些序列张量是如何被循环模型处理的（Move to the `LSTM` notebook and see how these sequence tensors are processed by a recurrent model.）